[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/Introduction-to-Machine-Learning-notebooks/blob/master/tree_basics.ipynb)

# Decision trees, in pictures

Run each cell with **Shift and Enter**, from the top. One cell sets everything
up; after that each cell is a single line with a single number in it. Three
questions are put along the way, and each answer is one click away.

## What is a decision tree?

A decision tree asks one measurement one question at a time. *Is the petal
narrower than 0.8 cm?* Whatever the answer, it either asks another question or
stops and names a kind of flower.

Because every question is about one measurement, every cut it makes is a
straight line at right angles to an axis. A tree therefore carves the picture
into rectangular boxes and gives one answer inside each box. That is the whole
method: the rest is choosing which question to ask.

In [ ]:
# Run this cell once. Every cell after it is one line.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

INK, AMBER, GREY = "#1e3a5f", "#b45309", "#64748b"
TINT = ["#dfe4ec", "#fbeddc", "#e3efe4"]

iris = load_iris()
X = iris.data[:, [2, 3]]
y = iris.target
LABELS = [iris.feature_names[2], iris.feature_names[3]]
NAMES = list(iris.target_names)

ARC_X, ARC_Y = make_moons(n_samples=400, noise=0.30, random_state=0)
FIT_X, HELD_X, FIT_Y, HELD_Y = train_test_split(
    ARC_X, ARC_Y, test_size=0.3, random_state=0, stratify=ARC_Y)


def _points(ax, data, label, marks="os^"):
    for k, mark in enumerate(marks[:len(set(label))]):
        ax.scatter(data[label == k, 0], data[label == k, 1], marker=mark,
                   s=16, color=[INK, AMBER, GREY][k], zorder=3)


def _zones(ax, model, data):
    pad = 0.4
    gx, gy = np.meshgrid(
        np.linspace(data[:, 0].min() - pad, data[:, 0].max() + pad, 300),
        np.linspace(data[:, 1].min() - pad, data[:, 1].max() + pad, 300))
    zone = model.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    n = len(np.unique(zone))
    ax.contourf(gx, gy, zone, levels=np.arange(-0.5, max(n, 2) + 0.5),
                colors=TINT[:max(n, 2)])


def idea():
    """A chain of questions on the left, and the boxes it cuts on the right."""
    fig, (a, b) = plt.subplots(1, 2, figsize=(9.5, 3.6))
    nodes = {"q1": (0.50, 0.86, "width < 0.8 ?"), "L": (0.22, 0.52, "setosa"),
             "q2": (0.74, 0.52, "width < 1.75 ?"),
             "LL": (0.58, 0.16, "versicolor"), "RR": (0.92, 0.16, "virginica")}
    leaves = {"L": 0, "LL": 1, "RR": 2}
    for key, (x, yy, text) in nodes.items():
        a.add_patch(plt.Rectangle(
            (x - 0.15, yy - 0.06), 0.30, 0.12, lw=1.0,
            facecolor=TINT[leaves[key]] if key in leaves else "white",
            edgecolor=AMBER if key in leaves else GREY))
        a.text(x, yy, text, ha="center", va="center", fontsize=9, color=INK)
    for one, two in (("q1", "L"), ("q1", "q2"), ("q2", "LL"), ("q2", "RR")):
        x0, y0, _ = nodes[one]
        x1, y1, _ = nodes[two]
        a.annotate("", xy=(x1, y1 + 0.07), xytext=(x0, y0 - 0.07),
                   arrowprops=dict(arrowstyle="-|>", color=GREY, lw=1.0))
    a.text(0.30, 0.70, "yes", fontsize=8, color=GREY)
    a.text(0.66, 0.70, "no", fontsize=8, color=GREY)
    a.set_xlim(-0.02, 1.12)
    a.set_ylim(0.03, 1.0)
    a.axis("off")
    a.set_title("two questions", fontsize=10)

    b.add_patch(plt.Rectangle((0, 0), 7, 0.8, facecolor=TINT[0]))
    b.add_patch(plt.Rectangle((0, 0.8), 7, 0.95, facecolor=TINT[1]))
    b.add_patch(plt.Rectangle((0, 1.75), 7, 0.85, facecolor=TINT[2]))
    for line in (0.8, 1.75):
        b.axhline(line, color=AMBER, lw=1.2)
    _points(b, X, y)
    b.set_xlim(0, 7)
    b.set_ylim(0, 2.6)
    b.set_xlabel(LABELS[0])
    b.set_ylabel(LABELS[1])
    b.set_title("the three boxes they cut", fontsize=10)
    plt.tight_layout()
    plt.show()


def flowers():
    """The 150 flowers, two measurements each."""
    fig, ax = plt.subplots(figsize=(5.6, 3.6))
    for k, mark in enumerate("os^"):
        ax.scatter(X[y == k, 0], X[y == k, 1], marker=mark, s=26,
                   color=[INK, AMBER, GREY][k], label=NAMES[k])
    ax.set_xlabel(LABELS[0])
    ax.set_ylabel(LABELS[1])
    ax.legend(frameon=False, fontsize=9)
    plt.show()


def cut(depth):
    """Fit a tree that deep on the flowers, and draw the boxes it makes."""
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X, y)
    fig, ax = plt.subplots(figsize=(5.6, 3.6))
    _zones(ax, tree, X)
    _points(ax, X, y)
    ax.set_xlabel(LABELS[0])
    ax.set_ylabel(LABELS[1])
    ax.set_title("depth %d, and it gets %.3f of the flowers right"
                 % (depth, tree.score(X, y)))
    plt.show()


def as_tree(depth):
    """The same tree, drawn as the chain of questions it is."""
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X, y)
    fig, ax = plt.subplots(figsize=(8.5, 4.0))
    drawn = plot_tree(tree, feature_names=LABELS, class_names=NAMES,
                      filled=True, impurity=False, fontsize=9, ax=ax)
    fig.canvas.draw()
    for art in drawn:
        box = art.get_bbox_patch()
        for k, kind in enumerate(NAMES):
            if box is not None and art.get_text().endswith("class = " + kind):
                box.set_facecolor(TINT[k])
                box.set_edgecolor(GREY)
    plt.show()


def depth_curve():
    """What depth does to the rows a tree was fitted on and the rows held back."""
    depths = range(1, 13)
    fitted = [DecisionTreeClassifier(max_depth=d, random_state=0)
              .fit(FIT_X, FIT_Y) for d in depths]
    on_fit = [t.score(FIT_X, FIT_Y) for t in fitted]
    on_held = [t.score(HELD_X, HELD_Y) for t in fitted]
    top = int(np.argmax(on_held))
    fig, ax = plt.subplots(figsize=(6.2, 3.6))
    ax.plot(depths, on_fit, color=GREY, lw=1.4, ls="--", marker="o", ms=4,
            label="rows it was fitted on")
    ax.plot(depths, on_held, color=INK, lw=1.6, marker="o", ms=4,
            label="rows held back")
    ax.plot([list(depths)[top]], [on_held[top]], marker="o", ms=10,
            color=AMBER, zorder=4)
    ax.set_xlabel("max_depth")
    ax.set_ylabel("share it gets right")
    ax.legend(frameon=False, fontsize=9, loc="upper left")
    plt.show()
    free = DecisionTreeClassifier(random_state=0).fit(FIT_X, FIT_Y)
    print("best on the rows held back: depth %d at %.3f"
          % (list(depths)[top], on_held[top]))
    print("with no limit at all:       %.3f fitted, %.3f held back, "
          "%d leaves, depth %d"
          % (free.score(FIT_X, FIT_Y), free.score(HELD_X, HELD_Y),
             free.get_n_leaves(), free.get_depth()))


def staircase(depth):
    """A boundary that runs at an angle, and the steps a tree makes of it."""
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(
        FIT_X, FIT_Y)
    fig, ax = plt.subplots(figsize=(5.6, 3.6))
    _zones(ax, tree, ARC_X)
    _points(ax, ARC_X, ARC_Y, marks="os")
    ax.set_xlabel("first measurement")
    ax.set_ylabel("second measurement")
    ax.set_title("depth %d, %.3f of the rows held back"
                 % (depth, tree.score(HELD_X, HELD_Y)))
    plt.show()

## 1. Two questions, three boxes

In [ ]:
idea()

On the left, two questions and three answers. On the right, the same two
questions drawn as what they do to the measurements: a horizontal line at
0.8 cm, another at 1.75 cm, and three bands.

Nothing here is fitted yet. This is what a tree *is*.

## 2. The flowers

150 iris flowers, 50 of each of three kinds. Each is measured twice, the
length and the width of its petal, in centimetres.

In [ ]:
flowers()

One kind sits alone in the bottom left. The other two touch along one edge,
and that edge is where every mistake below happens.

## 3. One question, one cut

A tree of depth 1 asks a single question and then answers.

In [ ]:
cut(depth=1)

The picture is cut in two by a flat line, because the question is about one
measurement. Everything below the line gets one answer and everything above it
gets another.

> **Question 1.** Change `cut(depth=1)` to `cut(depth=2)` and run the cell
> again. What appears, and what happens to the score?

<details>
<summary><b>Show the answer</b></summary>

A second cut appears, at a right angle to the first, and the picture now has
three boxes. The score goes from 0.667 to 0.960. Two questions tell three kinds
of flower apart almost perfectly, and the flowers still misread are the ones on
the edge where the two kinds touch.

</details>

## 4. The same tree, drawn as a tree

In [ ]:
as_tree(depth=2)

Each box is a question and the left branch is taken when the answer is yes.
`samples` is how many flowers reached that box. `value` counts them by kind, in
the order setosa, versicolor, virginica. `class` is the answer the box gives,
which is whichever kind is commonest in it.

## 5. How deep should it go?

The flowers are easy. These 400 rows are not: they lie in two interleaving
arcs with noise on them. 280 rows are used to fit the tree and 120 are held
back and never seen.

In [ ]:
depth_curve()

The dashed line is the share the tree gets right on the rows it was fitted on.
It climbs to 1.000, because a deep enough tree can put every row in a box of
its own. The solid line is the share on the rows held back, and it peaks and
then falls.

> **Question 2.** Read the two lines. At which depth is the tree best on rows
> it has never seen, and what does a deeper tree buy?

<details>
<summary><b>Show the answer</b></summary>

Depth 2, at 0.900. Past that the dashed line keeps climbing and the solid line
falls away to about 0.83: a deeper tree buys accuracy on rows it has already
seen and pays for it on rows it has not. Grown with no limit at all it reaches
1.000 on the rows it was fitted on and 0.825 on the rows held back.

</details>

## 6. A boundary at an angle

In [ ]:
staircase(depth=2)

The arcs are not separated by any horizontal or vertical line, and a tree has
nothing else to offer. It approximates the true boundary with a staircase.

> **Question 3.** Change `staircase(depth=2)` to `staircase(depth=10)`.
> How does the staircase change, and does the score follow?

<details>
<summary><b>Show the answer</b></summary>

The steps become many and narrow, and some of them fence off single points.
The score on the rows held back falls from 0.900 to 0.825. Finer steps follow
the rows the tree was fitted on, including the noise in them, and the noise is
different in the rows held back.

</details>

## 7. The mathematics

Everything above works without this section. It is here for a reader who wants
to know what "choosing which question to ask" means exactly.

A tree answers with one constant on each box:

$$\hat f(x) = \sum_m c_m \, \mathbf{1}\{x \in R_m\}$$

where $R_m$ are the boxes and $c_m$ is the commonest class inside box $m$. The
boxes are what has to be chosen, and they are chosen one cut at a time.

**Impurity.** A box holding a share $p_k$ of each class $k$ has Gini impurity

$$i(p) = 1 - \sum_k p_k^2$$

which is 0 when the box holds one class only and largest when the classes are
spread evenly.

**The gain of a cut.** Cutting column $j$ at threshold $t$ sends a share
$\lambda$ of the rows left and the rest right, and removes

$$\Delta(j,t) = i(p) - \lambda \, i(p_L) - (1-\lambda) \, i(p_R)$$

The tree tries every column and every threshold between two observed values,
takes the cut with the largest $\Delta$, and repeats inside each half. It never
looks back, which is why the tree it grows is a good tree and not the best one.

In [ ]:
root = 1 - sum((np.bincount(y) / len(y)) ** 2)
best = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X, y)
print("impurity before any cut: %.3f" % root)
print("the cut it chooses:      %s below %.2f"
      % (LABELS[best.tree_.feature[0]], best.tree_.threshold[0]))
print("impurity it removes:     %.3f" % (root - sum(
    best.tree_.n_node_samples[k] / len(y)
    * (1 - sum((best.tree_.value[k][0] / best.tree_.value[k][0].sum()) ** 2))
    for k in (1, 2))))

## What you can say now

- A decision tree is a chain of questions, each about one measurement, and the
  boxes it cuts have sides parallel to the axes.
- Inside a box every point gets the same answer, and that answer is the
  commonest class in the box.
- Depth is the one dial that matters most: too little and the tree cannot
  follow the data, too much and it follows the noise.
- A slanted boundary comes out as a staircase, however deep the tree goes.

The companion notebook, **Random forests, in pictures**, takes the last point
and does something about it.